# ARCHS4 CLAMP models: Effect of CLAMP_K (1% random sampling)

**Environment:** `clamp-analyses`

This notebook evaluates the effect of different CLAMP_K values on model results using 1% random sampling of ARCHS4 data.

Instead of estimating CLAMP_K via `num.pc()`, we fix it to: 50, 100, 250, 500, 750, 1000.

For each CLAMP_K value, 5 models are built with different random seeds (and thus different subsamples and SVDs).

Structure per seed:
1. Subsample ARCHS4 data (1% coverage, random sampling)
2. Compute SVD (shared across CLAMP_K values for the same seed)
3. For each CLAMP_K: run CLAMPbase, then CLAMPfull with BP prior

## Load libraries

In [4]:
if (!requireNamespace("CLAMP", quietly = TRUE)) {
    REPO_PATH <- "/home/msubirana/Documents/pivlab/CLAMP" 
    remotes::install_local(REPO_PATH, force = TRUE, dependencies = FALSE)
}

library(bigstatsr)
library(data.table)
library(dplyr)
library(rsvd)
library(glmnet)
library(Matrix)
library(knitr)
library(here)
library(CLAMP)
library(rhdf5)

source(here("config.R"))

## Configuration

In [5]:
# Base output directory
base_output_dir <- config$ARCHS4$DATASET_FOLDER

# Coverage level
coverage <- 0.01
coverage_pct <- coverage * 100

# CLAMP_K values to test
clamp_k_values <- c(50, 100, 250, 500, 750, 1000)

# Data path for priors
data_path <- here::here('data/archs4')

# SVD parameters
N_CORES <- 4

# Seeds for 5 runs
base_seed <- config$ARCHS4$CLAMP_PARAMS$RANDOM_SVD_SEED
seeds <- base_seed + 0:4
n_runs <- length(seeds)

# Output directory
output_base <- file.path(base_output_dir, "clamp_k_effect_01")
dir.create(output_base, showWarnings = FALSE, recursive = TRUE)

message("CLAMP_K values: ", paste(clamp_k_values, collapse = ", "))
message("Seeds: ", paste(seeds, collapse = ", "))
message("Coverage: ", coverage_pct, "%")
message("Output: ", output_base)

CLAMP_K values: 50, 100, 250, 500, 750, 1000

Seeds: 123, 124, 125, 126, 127

Coverage: 1%

Output: /home/msubirana/Documents/pivlab/clamp-analyses/output/archs4/clamp_k_effect_01



## Load preprocessed data

In [6]:
# Load metadata
meta <- readRDS(file.path(base_output_dir, "metadata_filtered.rds"))
n_genes_thin <- meta$n_genes_thin
n_samples_total <- meta$n_samples
archs4_genes <- meta$gene_symbols_thin

# Load sample names
all_samples <- readRDS(file.path(base_output_dir, "all_samples.rds"))
sample_names_total <- all_samples[seq_len(n_samples_total)]

# Load BP pathway
BP_pathMat <- readRDS(file.path(data_path, "BP_pathMat.rds"))
BP_matched <- getMatchedPathwayMat(BP_pathMat, archs4_genes)
message("Loaded and matched BP pathway matrix")

# Load FBM
fbm_file <- file.path(base_output_dir, "fbm")
output_file <- paste0(fbm_file, "_filtered")

archs4_fbm_filt <- FBM(
  nrow        = n_genes_thin,
  ncol        = n_samples_total,
  backingfile = output_file,
  create_bk   = FALSE
)

message("Loaded FBM with ", n_genes_thin, " genes and ", n_samples_total, " samples")

There are 11936 genes in the intersection between data and prior

Removing 2059 pathways

Loaded and matched BP pathway matrix

Loaded FBM with 18423 genes and 605614 samples



## Run models: 5 seeds x 6 CLAMP_K values

For each seed: subsample → SVD (once), then loop over CLAMP_K values for CLAMPbase + CLAMPfull_BP.

In [9]:
n_samples_target <- round(n_samples_total * coverage)

SVD_K <- min(n_samples_target - 1, n_genes_thin - 1)

message("Target samples per run: ", n_samples_target, " (", coverage_pct, "% of ", n_samples_total, ")")
message("SVD K: ", SVD_K)

results_summary <- data.frame(
  run = integer(),
  seed = integer(),
  CLAMP_K = integer(),
  n_samples = integer(),
  stringsAsFactors = FALSE
)

for (run_idx in seq_len(n_runs)) {
  current_seed <- seeds[run_idx]
  message("\n", strrep("=", 60))
  message("SEED ", run_idx, "/", n_runs, " (", current_seed, ")")
  message(strrep("=", 60))
  
  seed_dir <- file.path(output_base, paste0("seed_", run_idx))
  dir.create(seed_dir, showWarnings = FALSE, recursive = TRUE)
  
  # --- Sample selection ---
  set.seed(current_seed)
  sample_idx <- sort(sample(seq_len(n_samples_total), n_samples_target))
  n_samples <- length(sample_idx)
  sample_names <- sample_names_total[sample_idx]
  
  message("Selected ", n_samples, " samples")
  
  saveRDS(list(
    run = run_idx,
    seed = current_seed,
    coverage = coverage,
    n_samples = n_samples,
    sample_idx = sample_idx,
    sample_names = sample_names,
    sampling_method = "random_sampling"
  ), file = file.path(seed_dir, "subsample_info.rds"))
  
  # --- Create subsampled FBM ---
  message("Creating subsampled FBM...")
  fbm_sub_file <- file.path(seed_dir, "fbm_subsampled")
  
  Y_sub <- big_copy(
    archs4_fbm_filt,
    ind.col = sample_idx,
    backingfile = fbm_sub_file
  )
  
  # --- SVD (once per seed) ---
  message("Computing SVD (k=", SVD_K, ")...")
  
  if (N_CORES > 1) {
    options(bigstatsr.check.parallel.blas = FALSE)
    blas_nproc <- getOption("default.nproc.blas")
    options(default.nproc.blas = NULL)
  }
  
  svd_result <- big_randomSVD(Y_sub, k = SVD_K, ncores = N_CORES)
  
  if (N_CORES > 1) {
    options(bigstatsr.check.parallel.blas = TRUE)
    options(default.nproc.blas = blas_nproc)
  }
  
  valid_idx <- which(!is.nan(svd_result$d))
  svd_result$d <- svd_result$d[valid_idx]
  svd_result$u <- svd_result$u[, valid_idx, drop = FALSE]
  svd_result$v <- svd_result$v[, valid_idx, drop = FALSE]
  
  saveRDS(svd_result, file = file.path(seed_dir, "svd.rds"))
  message("SVD done: ", length(svd_result$d), " valid components")
  
  # --- Loop over CLAMP_K values ---
  for (CLAMP_K in clamp_k_values) {
    message("\n--- CLAMP_K = ", CLAMP_K, " (seed ", run_idx, ") ---")
    
    k_dir <- file.path(seed_dir, paste0("clamp_k_", CLAMP_K))
    dir.create(k_dir, showWarnings = FALSE, recursive = TRUE)
    
    saveRDS(CLAMP_K, file = file.path(k_dir, "CLAMP_K.rds"))
    
    # CLAMPbase
    message("Running CLAMPbase...")
    baseRes <- CLAMPbase(
      Y = Y_sub,
      svdres = svd_result,
      trace = TRUE,
      clamp_k = CLAMP_K
    )
    
    baseRes$Z <- data.frame(baseRes$Z)
    rownames(baseRes$Z) <- archs4_genes
    baseRes$B <- data.frame(baseRes$B)
    colnames(baseRes$B) <- sample_names
    
    saveRDS(baseRes, file = file.path(k_dir, "CLAMPbase.rds"))
    
    model_dir <- file.path(k_dir, "CLAMPbase")
    dir.create(model_dir, showWarnings = FALSE, recursive = TRUE)
    write.csv(baseRes$B, file.path(model_dir, "B.csv"))
    write.csv(baseRes$Z, file.path(model_dir, "Z.csv"))
    
    # CLAMPfull with BP prior
    message("Running CLAMPfull with BP prior...")
    fullRes <- CLAMPfull(
      Y = Y_sub,
      svdres = svd_result,
      priorMat = BP_matched,
      clamp.base.result = baseRes,
      use_cpp = TRUE,
      trace = TRUE,
      clamp_k = CLAMP_K
    )
    
    fullRes$Z <- data.frame(fullRes$Z)
    rownames(fullRes$Z) <- archs4_genes
    fullRes$B <- data.frame(fullRes$B)
    colnames(fullRes$B) <- sample_names
    fullRes$summary <- fullRes$summary %>%
      dplyr::rename(LV = LV_index) %>%
      dplyr::mutate(LV = paste0('LV', LV))
    
    saveRDS(fullRes, file = file.path(k_dir, "CLAMPfull_BP.rds"))
    
    model_dir <- file.path(k_dir, "CLAMPfull_BP")
    dir.create(model_dir, showWarnings = FALSE, recursive = TRUE)
    write.csv(fullRes$B, file.path(model_dir, "B.csv"))
    write.csv(fullRes$Z, file.path(model_dir, "Z.csv"))
    write.csv(fullRes$summary, file.path(model_dir, "summary.csv"))
    
    # Store summary
    results_summary <- rbind(results_summary, data.frame(
      run = run_idx,
      seed = current_seed,
      CLAMP_K = CLAMP_K,
      n_samples = n_samples
    ))
    
    rm(baseRes, fullRes)
    gc()
  }
  
  rm(Y_sub, svd_result)
  gc()
}

message("\n", strrep("=", 60))
message("All runs completed!")
message(strrep("=", 60))

Target samples per run: 6056 (1% of 605614)

SVD K: 6055





SEED 1/5 (123)


Selected 6056 samples

Creating subsampled FBM...

Computing SVD (k=6055)...

SVD done: 6054 valid components


--- CLAMP_K = 50 (seed 1) ---

Running CLAMPbase...

****

CLAMP k is set to 50

L1 is set to 125.207441857591

L2 is set to 375.622325572772

Progress 1 / 200 | Bdiff=0.356647, minCor=0.916221

Progress 2 / 200 | Bdiff=0.024771, minCor=0.967995

Progress 3 / 200 | Bdiff=0.010120, minCor=0.981919

Progress 4 / 200 | Bdiff=0.006792, minCor=0.985494

Progress 5 / 200 | Bdiff=0.005196, minCor=0.988777

Progress 6 / 200 | Bdiff=0.004334, minCor=0.991789

Progress 7 / 200 | Bdiff=0.003816, minCor=0.994210

Progress 8 / 200 | Bdiff=0.003466, minCor=0.994480

Progress 9 / 200 | Bdiff=0.003200, minCor=0.994801

Progress 10 / 200 | Bdiff=0.002970, minCor=0.995399

Progress 11 / 200 | Bdiff=0.002754, minCor=0.996171

Progress 12 / 200 | Bdiff=0.002539, minCor=0.996007

Progress 13 / 200 | Bdiff=0.002311, minCor=0.996041

Progress 14 / 200 | Bdiff=0.002073, minCor=0.9

## Summary

In [10]:
# Save summary
write.csv(results_summary, file.path(output_base, "results_summary.csv"), row.names = FALSE)

kable(results_summary)
message("Results saved to: ", output_base)



| run| seed| CLAMP_K| n_samples|
|---:|----:|-------:|---------:|
|   1|  123|      50|      6056|
|   1|  123|     100|      6056|
|   1|  123|     250|      6056|
|   1|  123|     500|      6056|
|   1|  123|     750|      6056|
|   1|  123|    1000|      6056|
|   2|  124|      50|      6056|
|   2|  124|     100|      6056|
|   2|  124|     250|      6056|
|   2|  124|     500|      6056|
|   2|  124|     750|      6056|
|   2|  124|    1000|      6056|
|   3|  125|      50|      6056|
|   3|  125|     100|      6056|
|   3|  125|     250|      6056|
|   3|  125|     500|      6056|
|   3|  125|     750|      6056|
|   3|  125|    1000|      6056|
|   4|  126|      50|      6056|
|   4|  126|     100|      6056|
|   4|  126|     250|      6056|
|   4|  126|     500|      6056|
|   4|  126|     750|      6056|
|   4|  126|    1000|      6056|
|   5|  127|      50|      6056|
|   5|  127|     100|      6056|
|   5|  127|     250|      6056|
|   5|  127|     500|      6056|
|   5|  

Results saved to: /home/msubirana/Documents/pivlab/clamp-analyses/output/archs4/clamp_k_effect_01

